# Talking to an LLM from Python

A large language model (LLM) takes text in and sends text back.  We will run a
small open-source model, **Llama 3.2**, on our own computers using **Ollama**:
it is free, needs no account, and nothing leaves your machine.  Your Python code
talks to it the same way it would talk to ChatGPT or Claude over the internet.

Run each cell, but try to **predict what will happen before you run it**.

## Setup (once)

1. Install Ollama from https://ollama.com/download and start the app.
2. In a terminal, download the model (about 2 GB):  `ollama pull llama3.2`
3. Install the Python package:  `uv pip install ollama`  (or `pip install ollama`)

**Using Google Colab instead of your own computer?**  Run the next cell first.
It installs Ollama on Colab's computer and downloads the model.  Colab erases
everything when the session ends, so you have to re-run it every time.
On your own computer it just prints a message.

In [ ]:
import sys, subprocess, time

if "google.colab" in sys.modules:
    subprocess.run("apt-get install -y zstd > /dev/null && "
                   "curl -fsSL https://ollama.com/install.sh | sh", shell=True)
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    subprocess.run(["ollama", "pull", "llama3.2"])
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ollama"])
else:
    print("Not on Colab: nothing to do here.")

Check that Python can see the model:

In [ ]:
import ollama

for m in ollama.list()["models"]:
    print(m["model"])

You should see `llama3.2:latest`.  If you get an error instead, the Ollama
app is not running or the model has not been downloaded.

## 1. Your first message

In [ ]:
reply = ollama.chat(
    model="llama3.2",
    messages=[{"role": "user", "content": "In one sentence, what is a CSV file?"}],
)
print(reply["message"]["content"])

Run it again.  You will probably get a different answer: the model chooses
its words with some randomness, so its output varies from run to run.

## 2. Messages are a list of dictionaries

Each message is a dictionary with a `"role"` and a `"content"`.  There are three
roles:

- `"system"`: standing instructions from the **programmer** about how the model
  should behave for the whole conversation (the person chatting never sees it)
- `"user"`: what the person types
- `"assistant"`: what the model said earlier

Why a separate system message?  When you build an app on top of an LLM, *you*
set the rules, not the user.  Here is a course help bot whose rule is that it
never writes a student's solution for them.  The student asks for the solution
anyway:

In [ ]:
system_msg = {"role": "system", "content":
    "You are a teaching assistant for an intro Python course. "
    "Help students find their own bugs by asking guiding questions. "
    "Never write the complete solution for them."}

user_msg = {"role": "user", "content":
    "Write a function load_words(filename) that returns a list of the words in the file."}

with_rules = ollama.chat(model="llama3.2", messages=[system_msg, user_msg])
without    = ollama.chat(model="llama3.2", messages=[user_msg])

print("WITH the system message:\n", with_rules["message"]["content"])
print("\nWITHOUT it:\n", without["message"]["content"])

The student's question is the same both times; only the system message
changed.  Compare the two replies.  A small model like Llama 3.2 sometimes
breaks the rule anyway: a system message is a strong request, not a
guarantee.

## 3. The model has no memory

In [ ]:
r1 = ollama.chat(model="llama3.2",
                 messages=[{"role": "user", "content": "My name is Eric."}])
r2 = ollama.chat(model="llama3.2",
                 messages=[{"role": "user", "content": "What is my name?"}])
print(r2["message"]["content"])

Each call starts from scratch.  A chat app seems to remember because it sends
the **whole conversation** every time.  We can do the same by keeping a list of
messages and appending to it:

In [ ]:
messages = [{"role": "user", "content": "My name is Eric."}]
reply = ollama.chat(model="llama3.2", messages=messages)
messages.append({"role": "assistant", "content": reply["message"]["content"]})

messages.append({"role": "user", "content": "What is my name?"})
reply = ollama.chat(model="llama3.2", messages=messages)
print(reply["message"]["content"])
print(len(messages), "messages sent")

## 4. A reply is just a string

If you need something specific, ask for it, and then **check** what you got.
Models do not always follow instructions.

In [ ]:
reply = ollama.chat(model="llama3.2", messages=[
    {"role": "user",
     "content": "Give me one 5-letter English word. Reply with only the word."}
])
word = reply["message"]["content"].strip().lower()
print(repr(word))
print("exactly 5 letters?", len(word) == 5 and word.isalpha())

## 5. Where LLMs struggle: individual letters

In [ ]:
reply = ollama.chat(model="llama3.2", messages=[
    {"role": "user",
     "content": "How many times does the letter r appear in the word strawberry?"}
])
print("LLM says:   ", reply["message"]["content"])
print("Python says:", "strawberry".count("r"))

LLMs read text in chunks called **tokens**, not one letter at a time, so
questions about spelling and letters are a weak spot.  Try a few words of your
own.  Keep this in mind for Wordle, which is a game entirely about letters.

## 6. Saving the results to a file

Let's ask for a 5-letter word five times and record what happens in a CSV
file, so we can look at it later.

In [ ]:
import csv

question = "Give me one 5-letter English word. Reply with only the word."

with open("llm_words.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["attempt", "reply", "five_letters"])
    for attempt in range(1, 6):
        reply = ollama.chat(model="llama3.2",
                            messages=[{"role": "user", "content": question}])
        text = reply["message"]["content"].strip().lower()
        writer.writerow([attempt, text, len(text) == 5 and text.isalpha()])

with open("llm_words.csv") as f:
    print(f.read())

How often did it follow the instruction?  Did it repeat itself?  We will use
this kind of record-keeping to compare the LLM with our own Wordle bot.